In [4]:
import pandas as pd

matches = pd.read_csv("data/gold_match.csv")

# --- 1. Calculate Win Percentage ---
home_games = matches[['match_id', 'home_team', 'winner']].rename(columns={'home_team': 'team'})
home_games['is_win'] = home_games['winner'] == 'home'

away_games = matches[['match_id', 'away_team', 'winner']].rename(columns={'away_team': 'team'})
away_games['is_win'] = away_games['winner'] == 'away'

all_games = pd.concat([home_games, away_games], ignore_index=True)
win_stats = all_games.groupby('team').agg(
    total_games=('match_id', 'count'),
    total_wins=('is_win', 'sum')
)

# --- FILTER: Keep only teams with 3 or more matches ---
win_stats = win_stats[win_stats['total_games'] >= 3].copy()

win_stats['win_percentage'] = (win_stats['total_wins'] / win_stats['total_games']) * 100


# --- 2. Calculate Average Goal Difference ---
home_stats = matches[['home_team', 'goals_home_ft', 'goals_away_ft']].rename(
    columns={'home_team': 'team', 'goals_home_ft': 'goals_for', 'goals_away_ft': 'goals_against'})

away_stats = matches[['away_team', 'goals_away_ft', 'goals_home_ft']].rename(
    columns={'away_team': 'team', 'goals_away_ft': 'goals_for', 'goals_home_ft': 'goals_against'})

all_goals = pd.concat([home_stats, away_stats], ignore_index=True)
goal_stats = all_goals.groupby('team').agg(
    avg_goals_for=('goals_for', 'mean'),
    avg_goals_against=('goals_against', 'mean')
)
goal_stats['avg_goal_diff'] = goal_stats['avg_goals_for'] - goal_stats['avg_goals_against']


# --- 3. Merge and Calculate Team Strength ---
# Use inner join to ensure we only keep teams that passed the >= 3 matches filter
team_metrics = pd.merge(
    win_stats[['total_games', 'win_percentage']], 
    goal_stats[['avg_goal_diff']], 
    left_index=True, 
    right_index=True,
    how='inner'
).reset_index()

# Normalize Goal Difference (0 to 100 scale)
min_goal = team_metrics['avg_goal_diff'].min()
max_goal = team_metrics['avg_goal_diff'].max()
team_metrics['goal_strength'] = ((team_metrics['avg_goal_diff'] - min_goal) / (max_goal - min_goal)) * 100

# Normalize Win Percentage (0 to 100 scale)
min_win = team_metrics['win_percentage'].min()
max_win = team_metrics['win_percentage'].max()
team_metrics['win_strength'] = ((team_metrics['win_percentage'] - min_win) / (max_win - min_win)) * 100

# Calculate overall Team Strength
team_metrics['team_strength'] = (team_metrics['goal_strength'] + team_metrics['win_strength']) / 2


# --- 4. Format and Sort Final Output ---
final_table = team_metrics[['team', 'total_games', 'team_strength']].copy()
final_table = final_table.rename(columns={'total_games': 'amount of matches'})
final_table['team_strength'] = final_table['team_strength'].round(2)

final_table_sorted = final_table.sort_values(by='team_strength', ascending=False).reset_index(drop=True)

print("Team strength is based on 2 features:")
print("- Win percentage")
print("- Goal difference")
print("The strength is normalized between 0 and 100")
print()
print("Only teams with 3 or more matches are included")
final_table_sorted

Team strength is based on 2 features:
- Win percentage
- Goal difference
The strength is normalized between 0 and 100

Only teams with 3 or more matches are included


,team,amount of matches,team_strength
0,Union Saint-Gilloise,8,91.82
1,Club Brugge,8,86.36
2,Genk,8,66.82
3,Cercle Brugge,8,58.64
4,Anderlecht,8,53.64
5,Antwerp,7,47.40
6,Eupen,4,43.64
7,Westerlo,12,36.67
8,Mechelen,12,34.24
9,Gent,10,27.82
